# Detecting Outliers in Tabular Data with Isolation Forest on MLServe.com
---

This notebook demonstrates how to train, deploy, and query an unsupervised anomaly detection model using the MLServe.com SDK.

We’ll generate synthetic tabular data with a mix of numeric and categorical features, train an Isolation Forest model inside a scikit-learn pipeline, and deploy it as a standard MLServe.com endpoint.

The deployed model can be used to flag anomalous rows or transactions (e.g. fraud, sensor drift, abnormal user behavior).

## Workflow overview

1. Generate synthetic data with both normal and anomalous samples.
2. Preprocess data using standard encoders and scalers.
3. Train an `IsolationForest` model to identify outliers.
4. Deploy the model to MLServe.com via the SDK.
5. Predict anomaly scores for new samples in real time.

In [6]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
import json
from sklearn.datasets import make_classification
import os
from mlserve_sdk.client import MLServeClient
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
np.random.seed(42)
n_samples = 2000
n_outliers = 100

# --- numerical features ---
X_num, _ = make_classification(
    n_samples=n_samples,
    n_features=4,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42
)
df = pd.DataFrame(X_num, columns=["feature1", "feature2", "feature3", "feature4"])

# --- categorical feature ---
df["category"] = np.random.choice(["A", "B", "C"], n_samples)

# --- inject outliers ---
outliers = pd.DataFrame({
    "feature1": np.random.uniform(-10, 10, n_outliers),
    "feature2": np.random.uniform(-10, 10, n_outliers),
    "feature3": np.random.uniform(-10, 10, n_outliers),
    "feature4": np.random.uniform(-10, 10, n_outliers),
    "category": np.random.choice(["A", "B", "C"], n_outliers)
})
df = pd.concat([df, outliers], ignore_index=True)
print("✅ Data shape:", df.shape)

✅ Data shape: (2100, 5)


In [3]:
numeric_features = ["feature1", "feature2", "feature3", "feature4"]
categorical_features = ["category"]
all_features = numeric_features + categorical_features

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

model = IsolationForest(
    n_estimators=200,
    contamination=0.05,  # expected fraction of outliers
    random_state=42
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

pipeline.fit(df)
print("✅ IsolationForest trained")

✅ IsolationForest trained


In [25]:
test_sample = df.sample(5)
scores = pipeline.decision_function(test_sample)
preds = pipeline.predict(test_sample)  # -1 = outlier, 1 = normal

results = pd.DataFrame({
    "score": scores.round(3),
    "prediction": preds
})
print("\n🔍 Outlier predictions:")
print(pd.concat([test_sample.reset_index(drop=True), results], axis=1))


🔍 Outlier predictions:
   feature1  feature2  feature3  feature4 category  score  prediction
0 -6.516133  4.616589  5.868145  3.431169        B -0.150          -1
1  1.311162  0.864070  1.011703  1.094067        C  0.092           1
2 -0.302885  1.346100  0.701922  0.574111        A  0.117           1
3 -1.632004  4.657978  5.995622  4.743605        C -0.121          -1
4  1.871074 -0.850884 -0.550552 -0.445248        C  0.044           1


In [8]:
USERNAME = os.getenv("USERNAME")
TOKEN = os.getenv("TOKEN")

client = MLServeClient()
client.login(USERNAME, TOKEN)

In [9]:
try:
    lv = client.get_latest_version("outlier_detector")
    next_version = lv["next_version"]
except:
    next_version = "v1"

print("Next version:", next_version)

Next version: v2


In [ ]:
# Deploy model
client.deploy(
    model=pipeline,
    name="outlier_detector",
    version='v1',
    features=all_features,
    background_df=df.sample(100),
    task_type='outlier_detection',
    metrics={}
)

In [26]:
# --- Predict on new data ---
TEST_DATA = {
    "features": all_features,
    "inputs": test_sample.to_dict(orient="records")
}

preds = client.predict("outlier_detector", 'v1', TEST_DATA)
print("\nPredictions:", preds["predictions"])


Predictions: [-1, 1, 1, -1, 1]
